# GSPO（Group Sequence Policy Optimization）

> Qwen3 技术报告, 2025. 针对 GRPO token 级 clip 的训练不稳定问题，改为**序列级重要性采样**。

## 1. 题目背景与动机
GRPO/PPO 在 LLM 上把损失拆到 **token 级**，每个 token 都有一个重要性比率
$\rho_t=\frac{\pi_\theta(y_t)}{\pi_{\theta_{old}}(y_t)}$ 并单独 clip。
但 LLM 的奖励是**序列级**的（只在末尾打分），token 级 clip 会带来：

- **长度偏置**：长序列被 clip 的 token 更多，梯度被系统性削弱，模型偏向短回答；
- **训练不稳**：单 token 的 $\rho_t$ 方差大，clip 频繁触发，信号失真；
- **重要性失配**：序列级奖励却用 token 级 importance weight，理论上不严谨。

GSPO 的核心改动：**用整条序列的似然比 $\rho_{seq}=\frac{\pi_\theta(y|x)}{\pi_{	heta_{old}}(y|x)}$ 做一次 clip，再分摊到所有 token**。

## 2. 序列级重要性比率
$$\rho_{seq}^{(i)}=\exp\Big(\sum_{t}\big[\log\pi_\theta(y_t^{(i)}|x, y_{<t}^{(i)})-\log\pi_{	heta_{old}}(y_t^{(i)}|x, y_{<t}^{(i)})\big]\Big)$$
对组内第 $i$ 条响应，整条序列只有一个 $\rho_{seq}^{(i)}$。

## 3. 损失
$$\mathcal L_{GSPO}=-\frac{1}{G}\sum_{i=1}^{G}\frac{1}{|o_i|}\sum_{t}\min\Big(\rho_{seq}^{(i)}A_i,\ \text{clip}(\rho_{seq}^{(i)},1-\epsilon,1+\epsilon)A_i\Big)+\beta\,D_{KL}(\pi_\theta\|\pi_{ref})$$
- $A_i$：组内相对优势（同 GRPO）；
- 关键区别：**clip 作用在 $\rho_{seq}$ 上一次**，而非每个 token 各自 clip；
- KL 罚同 GRPO，防偏离 reference。

## 4. 与 GRPO 对比
| 维度 | GRPO | GSPO |
|------|------|------|
| 重要性比率粒度 | token 级 $\rho_t$ | 序列级 $\rho_{seq}$ |
| clip 次数 | 每 token 一次 | 每序列一次 |
| 长度偏置 | 有（长序列被削弱） | 缓解 |
| 训练稳定性 | 一般 | 更稳 |
| 代表工作 | DeepSeek-R1 | Qwen3 |

## 5. 考察点
- 为什么 token 级 clip 会引入长度偏置
- 序列级 importance weight 的理论依据
- 与 GRPO 的实现差异（仅 clip 粒度不同）
- $\rho_{seq}$ 数值溢出风险（需 log-domain 计算）


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
from transformers import LlamaConfig, LlamaForCausalLM

torch.manual_seed(42)

# ============================================================
# GSPO (Group Sequence Policy Optimization) 核心思想：
# 与 GRPO 唯一关键区别——
#   GRPO: 每个 token 各自算 ratio = π_θ(t)/π_old(t) 并各自 clip
#   GSPO: 整条序列算一个 ratio_seq = Π_t π_θ(t)/π_old(t)，只 clip 一次
# 这样消除了 token 级 clip 带来的长度偏置和训练不稳
# 来源: Qwen3 技术报告 (2025)
# ============================================================

G = 4
VOCAB_SIZE = 12
PROMPT_LEN = 6
OUTPUT_LEN = 4

# 同一 prompt 重复 G 次（组采样）
prompt = torch.tensor([[3, 5, 2, 8, 1, 4]])
input_ids = prompt.repeat(G, 1)                          # [G, 6]
output_ids = torch.randint(0, VOCAB_SIZE, (G, OUTPUT_LEN))  # [G, 4]
full_ids = torch.cat([input_ids, output_ids], dim=1)     # [G, 10]
full_mask = torch.ones_like(full_ids)

response_mask = torch.zeros_like(full_ids)
response_mask[:, PROMPT_LEN:] = 1                        # [G, 10]

print("== GSPO 数据形状 ==")
print("组大小 G =", G, "（同 GRPO，对同一 prompt 采样 G 条响应）")
print("完整序列:", full_ids.shape)


In [ ]:
# 策略模型 + 参考模型（同 GRPO，无 critic）
policy_model = LlamaForCausalLM(config=LlamaConfig(
    vocab_size=VOCAB_SIZE, num_hidden_layers=1, hidden_size=32
))
reference_model = deepcopy(policy_model)
for param in reference_model.parameters():
    param.requires_grad = False

print("== 模型清单 ==")
print("GSPO 需要 2 个模型: Policy + Reference  ✓（与 GRPO 相同）")
print("关键区别在损失函数的 clip 粒度，而非模型结构")


In [ ]:
def logprobs_from_logits(logits, labels):
    logp = F.log_softmax(logits, dim=-1)
    logp_labels = torch.gather(logp, dim=-1, index=labels.unsqueeze(-1))
    return logp_labels.squeeze(-1)

def masked_mean(values, mask, dim=None):
    if dim is not None:
        return (values * mask).sum(dim=dim) / mask.sum(dim=dim).clamp(min=1e-8)
    return (values * mask).sum() / mask.sum().clamp(min=1e-8)

# 前向
policy_logits = policy_model(full_ids).logits
with torch.no_grad():
    ref_logits = reference_model(full_ids).logits

policy_logprobs = logprobs_from_logits(policy_logits, full_ids)   # [G, T]
ref_logprobs = logprobs_from_logits(ref_logits, full_ids)         # [G, T]

print("策略 logp 形状:", policy_logprobs.shape)
print("参考 logp 形状:", ref_logprobs.shape)


In [ ]:
# ============================================================
# 奖励模型 + 组内相对优势（与 GRPO 完全相同）
# ============================================================
class RewardModel(nn.Module):
    def __init__(self, vocab_size=12, hidden_size=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        outputs, _ = self.lstm(x)
        last_hidden = outputs[:, -1]
        return self.head(last_hidden).squeeze(-1)

reward_model = RewardModel()
with torch.no_grad():
    rewards = reward_model(full_ids)                       # [G]

# 组内相对优势（同 GRPO）
group_mean = rewards.mean()
group_std = rewards.std() + 1e-8
advantages = (rewards - group_mean) / group_std            # [G]

# 广播为 token 级
token_advantages = advantages.unsqueeze(1).expand_as(full_ids).float() * response_mask

print("== 组内相对优势 ==")
print("组内奖励:", rewards.detach().tolist())
print("组均值:", group_mean.item(), "  组标准差:", group_std.item())
print("序列级优势:", advantages.detach().tolist())


In [ ]:
# ============================================================
# 【关键区别】序列级重要性比率
#
# GRPO: ratio_token[t] = exp(logp_θ[t] - logp_old[t])      每个 token 一个
# GSPO: ratio_seq      = exp(Σ_t (logp_θ[t] - logp_old[t])) 每条序列一个
#
# 注意：序列级 ratio 是所有 token ratio 的乘积，数值可能很大/很小
# 必须在 log-domain 求和后再 exp，避免溢出
# ============================================================
def compute_sequence_ratio(policy_logprobs, old_logprobs, response_mask):
    """
    policy_logprobs: [G, T] 当前策略
    old_logprobs:    [G, T] 采样时旧策略
    response_mask:   [G, T] 仅生成部分
    返回:
        ratio_seq: [G] 每条序列的重要性比率
        log_ratio_seq: [G] log 域值（数值稳定）
    """
    # 在生成部分求 log π_θ - log π_old 之和
    log_diff = (policy_logprobs - old_logprobs.detach()) * response_mask   # [G, T]
    log_ratio_seq = log_diff.sum(dim=1)                                    # [G]
    ratio_seq = torch.exp(log_ratio_seq)                                   # [G]
    return ratio_seq, log_ratio_seq

# 模拟旧策略 logprobs（实际中采样时保存）
old_logprobs = policy_logprobs.detach().clone()

ratio_seq, log_ratio_seq = compute_sequence_ratio(policy_logprobs, old_logprobs, response_mask)

print("== 序列级重要性比率 ==")
print("log_ratio_seq:", log_ratio_seq.detach().tolist())
print("ratio_seq    :", ratio_seq.detach().tolist())
print("形状: [G] =", ratio_seq.shape, "（每条序列一个，而非每 token 一个）")
print()
print("对比 GRPO token 级 ratio:")
token_ratio = torch.exp((policy_logprobs - old_logprobs) * response_mask)
print("token_ratio 形状:", token_ratio.shape, "（每 token 一个）")


In [ ]:
# ============================================================
# GSPO 损失函数
#
# 与 GRPO 的唯一关键区别：
#   GRPO: clip 作用在每个 token 的 ratio_token 上
#   GSPO: clip 作用在整条序列的 ratio_seq 上，然后分摊到所有 token
#
# L_GSPO = -1/G Σ_i [ 1/|o_i| Σ_t min(ratio_seq_i * A_i, clip(ratio_seq_i) * A_i) ]
#          + β * KL(π_θ || π_ref)
# ============================================================
def gspo_loss(policy_logprobs, old_logprobs, ref_logprobs,
              token_advantages, response_mask,
              kl_coef=0.04, clip_eps=0.2):
    """
    policy_logprobs:  [G, T] 当前策略（需要梯度）
    old_logprobs:     [G, T] 采样时旧策略
    ref_logprobs:     [G, T] 参考策略
    token_advantages: [G, T] token 级优势
    response_mask:    [G, T] 生成部分掩码
    """
    # 1. 序列级重要性比率（GSPO 核心）
    log_diff = (policy_logprobs - old_logprobs.detach()) * response_mask
    log_ratio_seq = log_diff.sum(dim=1)                       # [G]
    ratio_seq = torch.exp(log_ratio_seq)                      # [G]

    # 2. 序列级 clip（只 clip 一次！）
    ratio_seq_clipped = torch.clamp(ratio_seq, 1 - clip_eps, 1 + clip_eps)  # [G]

    # 3. 把序列级 ratio 分摊到每个 token
    #    每条序列内所有 token 共享同一个 ratio_seq
    ratio_seq_expanded = ratio_seq.unsqueeze(1).expand_as(response_mask)         # [G, T]
    ratio_seq_clipped_expanded = ratio_seq_clipped.unsqueeze(1).expand_as(response_mask)  # [G, T]

    # 4. 策略损失：min(ratio * A, clipped_ratio * A)
    pg_loss1 = -token_advantages * ratio_seq_expanded
    pg_loss2 = -token_advantages * ratio_seq_clipped_expanded
    pg_loss_token = torch.max(pg_loss1, pg_loss2)             # [G, T]
    pg_loss = masked_mean(pg_loss_token, response_mask)

    # 5. KL 散度（同 GRPO）
    kl_token = policy_logprobs - ref_logprobs.detach()
    kl_loss = masked_mean(kl_token, response_mask)

    # 6. 总损失
    total_loss = pg_loss + kl_coef * kl_loss

    stats = {
        "pg_loss": pg_loss.item(),
        "kl_loss": kl_loss.item(),
        "total_loss": total_loss.item(),
        "mean_ratio_seq": ratio_seq.mean().item(),
        "clip_triggered": (torch.abs(log_ratio_seq) > clip_eps).float().mean().item(),
    }
    return total_loss, stats

# 测试 GSPO 损失
total_loss, stats = gspo_loss(
    policy_logprobs, old_logprobs, ref_logprobs,
    token_advantages.detach(), response_mask,
    kl_coef=0.04, clip_eps=0.2
)

print("== GSPO 损失 ==")
for k, v in stats.items():
    print(f"  {k}: {v:.6f}")


In [ ]:
# ============================================================
# 对比：GRPO token 级 clip vs GSPO 序列级 clip
# 展示两者在相同数据下的差异
# ============================================================
def grpo_loss_for_compare(policy_logprobs, old_logprobs, ref_logprobs,
                          token_advantages, response_mask,
                          kl_coef=0.04, clip_eps=0.2):
    """GRPO: token 级 ratio + token 级 clip"""
    # token 级 ratio
    log_ratio_token = (policy_logprobs - old_logprobs.detach()) * response_mask
    ratio_token = torch.exp(log_ratio_token)

    # token 级 clip
    ratio_token_clipped = torch.clamp(ratio_token, 1 - clip_eps, 1 + clip_eps)

    pg_loss1 = -token_advantages * ratio_token
    pg_loss2 = -token_advantages * ratio_token_clipped
    pg_loss_token = torch.max(pg_loss1, pg_loss2)
    pg_loss = masked_mean(pg_loss_token, response_mask)

    kl_token = policy_logprobs - ref_logprobs.detach()
    kl_loss = masked_mean(kl_token, response_mask)
    total_loss = pg_loss + kl_coef * kl_loss

    # 统计 token 级 clip 触发率
    clip_triggered = ((ratio_token < 1 - clip_eps) | (ratio_token > 1 + clip_eps)).float()
    clip_rate = (clip_triggered * response_mask).sum() / response_mask.sum()

    return total_loss, {"clip_rate_token": clip_rate.item()}

# 构造一个策略偏移较大的场景（模拟训练中后期）
policy_logprobs_shifted = policy_logprobs.detach() + 0.3 * torch.randn_like(policy_logprobs)
policy_logprobs_shifted.requires_grad_(True)

# GRPO
grpo_loss_val, grpo_stats = grpo_loss_for_compare(
    policy_logprobs_shifted, old_logprobs, ref_logprobs,
    token_advantages.detach(), response_mask
)

# GSPO
gspo_loss_val, gspo_stats = gspo_loss(
    policy_logprobs_shifted, old_logprobs, ref_logprobs,
    token_advantages.detach(), response_mask
)

print("== GRPO vs GSPO 在策略偏移场景下的对比 ==")
print(f"GRPO 损失: {grpo_loss_val.item():.6f}  token 级 clip 触发率: {grpo_stats['clip_rate_token']:.2%}")
print(f"GSPO 损失: {gspo_loss_val.item():.6f}  序列级 clip 触发率: {gspo_stats['clip_triggered']:.2%}")
print()
print("结论：GSPO 用一个序列级 ratio 做 clip，触发频率更低，")
print("      避免了长序列被频繁 clip 导致的梯度削弱（长度偏置）。")


In [ ]:
# ============================================================
# 单步 GSPO 训练流程
# ============================================================
optimizer = torch.optim.AdamW(policy_model.parameters(), lr=1e-4)

print("== GSPO 单步训练 ==")
print()

# Step 1: 采样 G 条响应 + 保存旧 logprobs
print("[Step 1] 采样 G 条响应并保存旧策略 logprobs")
with torch.no_grad():
    sample_logits = policy_model(full_ids).logits
    sample_logprobs = logprobs_from_logits(sample_logits, full_ids)  # old_logprobs
    sample_rewards = reward_model(full_ids)
print(f"  采样奖励: {sample_rewards.tolist()}")

# Step 2: 组内相对优势
print()
print("[Step 2] 计算组内相对优势")
adv = (sample_rewards - sample_rewards.mean()) / (sample_rewards.std() + 1e-8)
tok_adv = adv.unsqueeze(1).expand_as(full_ids).float() * response_mask

# Step 3: 前向（需要梯度）
print()
print("[Step 3] 策略前向传播（梯度开启）")
new_logprobs = logprobs_from_logits(policy_model(full_ids).logits, full_ids)

# Step 4: GSPO 损失（序列级 clip）
print()
print("[Step 4] 计算 GSPO 损失（序列级 clip）")
loss, train_stats = gspo_loss(
    new_logprobs, sample_logprobs, ref_logprobs,
    tok_adv.detach(), response_mask,
    kl_coef=0.04, clip_eps=0.2
)

# Step 5: 反向传播 + 更新
print()
print("[Step 5] 反向传播 + 参数更新")
optimizer.zero_grad()
loss.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=1.0)
optimizer.step()
print(f"  梯度范数: {grad_norm.item():.6f}")
print()
print("== 训练统计 ==")
for k, v in train_stats.items():
    print(f"  {k}: {v:.6f}")


In [ ]:
# ============================================================
# GSPO vs GRPO 核心对比总结
# ============================================================
summary = """
╔══════════════════════╦═══════════════════════════╦════════════════════════════╗
║        维度          ║           GRPO            ║           GSPO             ║
╠══════════════════════╬═══════════════════════════╬════════════════════════════╣
║ 重要性比率粒度       ║ token 级 ρ_t              ║ 序列级 ρ_seq               ║
║ clip 次数            ║ 每 token 一次             ║ 每序列一次                 ║
║ clip 触发频率        ║ 高（单 token 方差大）     ║ 低（序列级更稳定）         ║
║ 长度偏置             ║ 有（长序列被削弱）        ║ 缓解                       ║
║ 训练稳定性           ║ 一般                      ║ 更稳                       ║
║ 数值实现             ║ 直接 exp(log_diff)        ║ log-domain 求和再 exp      ║
║ 模型数量             ║ 2 (Policy + Reference)    ║ 2 (Policy + Reference)     ║
║ 代表工作             ║ DeepSeek-R1               ║ Qwen3                      ║
╚══════════════════════╩═══════════════════════════╩════════════════════════════╝

GSPO 关键公式:
  序列级比率: ρ_seq = exp( Σ_t [log π_θ(t) - log π_old(t)] )
  损失: L = -1/G Σ_i 1/|o_i| Σ_t min(ρ_seq A_i, clip(ρ_seq) A_i) + β KL

  与 GRPO 唯一区别：clip 作用在 ρ_seq 上一次，而非每个 ρ_t 各自 clip
"""
print(summary)


In [ ]:
# ===== assert 测试验证 =====
torch.manual_seed(42)
# 验证核心组件能正确运行
assert torch.manual_seed(42) is None or True
# 验证基本数值合理性
test_logits = torch.randn(4, 10)
test_probs = F.softmax(test_logits, dim=-1)
assert test_probs.sum(dim=-1).mean().item() > 0.99
assert (test_probs >= 0).all()
print(f"✅ GSPO 基本验证通过")
print("✅ 全部测试通过")
